# Experiment 2.1b-ii: Does Equivariance Violation Predict Per-Layer Muon Advantage?

---

## Theoretical Background

In the Muon-as-RG-Gauge-Fixing framework, the Newton-Schulz orthogonalization step
acts as a **gauge-fixing** operation: it projects gradient updates onto the orthogonal
group, removing the redundant "gauge degrees of freedom" that arise from the
reparametrization symmetry of deep linear networks.

Experiment 2.1b established a key surprise: **equivariance of Muon's Newton-Schulz
step breaks when the loss depends on data**. Specifically, if we conjugate the initial
weights of a target layer by orthogonal matrices $W_\ell \to R\,W_\ell\,S^\top$, the
trained output does NOT simply conjugate accordingly. The violation is systematic
and varies across layers.

This raises a critical follow-up question: **does the magnitude of equivariance
violation predict how much Muon helps a given layer?**

## Competing Hypotheses

Two competing interpretations exist:

| Hypothesis | Prediction | Mechanism |
|:-----------|:-----------|:----------|
| **(A) More violation = more benefit** | Positive correlation | Larger violation implies more gauge drift under SGD, so Muon's gauge-fixing has more work to do and yields larger improvement |
| **(B) More violation = less benefit** | Negative correlation | Larger violation means the gauge-fixing itself is imprecise at that layer, so Muon's advantage is diminished |

We operationalize "Muon advantage" as the **condition number ratio** $\kappa_{\text{SGD}} / \kappa_{\text{Muon}}$ at convergence: a ratio > 1 means Muon produces better-conditioned weight matrices.

## Experimental Design

- Train an 8-layer, 32x32 deep linear network with both SGD (momentum) and Muon
- For each layer $\ell$, measure:
  - **Equivariance violation**: relative norm difference between actual and expected outputs under random orthogonal conjugation
  - **Kappa advantage**: ratio of SGD's condition number to Muon's condition number at that layer
- Repeat over 5 random seeds and correlate violation with advantage across layers

## Key Tests

| Test | Criterion | Interpretation |
|:-----|:----------|:---------------|
| T1 | $r > 0$ | Positive correlation supports Hypothesis A |
| T3 | $|r| > 0.5$ | Strong correlation indicates the relationship is not merely noise |

## Environment Setup

Import NumPy for linear algebra operations. This experiment is entirely CPU-based
since we work with small 32x32 matrices in a deep linear network -- no GPU or
deep learning framework is needed.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

Define the hyperparameters and experimental setup.

**Architecture**: An 8-layer deep linear network with 32x32 weight matrices. Each layer
is initialized as $W_\ell = I + 0.1 \cdot \mathcal{N}(0,1)$, i.e., near-identity with
small perturbations. This is deep enough to exhibit non-trivial layer-dependent effects.

**Optimization**: SGD and Muon both use momentum (0.9). Muon uses 5 Newton-Schulz
iterations for orthogonalization. The learning rates differ (0.01 for Muon, 0.005 for
SGD) as each optimizer requires its own stable regime.

**Statistics**: 5 independent seeds provide cross-seed averaging to reduce noise in
the per-layer measurements.

In [ ]:
DIM = 32
DEPTH = 8
NUM_STEPS = 300
LR_MUON = 0.01
LR_SGD = 0.005
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")
print(f"  DEPTH = {DEPTH}")


## Core Algorithms

### Newton-Schulz Orthogonalization

The Newton-Schulz iteration computes the polar factor of a matrix via the recurrence:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^\top X_k)$$

Starting from $X_0 = M / \|M\|_F$, this converges cubically to the nearest orthogonal
matrix $U$ in the polar decomposition $M = U P$. This is the heart of Muon's
gauge-fixing: it strips away the positive-definite "stretch" factor $P$ and keeps
only the rotation $U$.

### Random Orthogonal Matrix Generation

We generate uniformly random orthogonal matrices via QR decomposition of Gaussian
matrices, with sign correction to ensure uniform Haar measure on $O(n)$.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def random_orthogonal(n, rng):
    A = rng.randn(n, n)
    Q, R = np.linalg.qr(A)
    D = np.diag(np.sign(np.diag(R)))
    return Q @ D

### Weight Initialization

Each layer is initialized as $W_\ell = I_{32} + 0.1\,\epsilon$ where $\epsilon \sim \mathcal{N}(0,1)$.
This near-identity initialization ensures:
1. The initial forward pass is close to a residual connection
2. Gradients flow cleanly through the depth-8 network at initialization
3. The initial condition number $\kappa(W_\ell) \approx 1$, so any divergence from 1 at convergence reflects optimizer-induced spectral shaping

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(DEPTH)]

## Forward Pass, Loss, and Gradient Computation

The network computes $\hat{Y} = W_L \cdots W_2 W_1 X$ with MSE loss
$\mathcal{L} = \frac{1}{2N}\sum_j \|\hat{y}_j - y_j\|^2$.

Gradients are computed by standard backpropagation through the linear chain.
For layer $\ell$, the gradient is:
$$\nabla_{W_\ell} \mathcal{L} = \delta_\ell \, a_{\ell-1}^\top$$
where $\delta_\ell$ is the error signal backpropagated from the output and
$a_{\ell-1}$ is the activation (pre-multiplication) at layer $\ell-1$.

Note: in a deep linear network, the loss landscape is non-convex despite
the linearity of each layer, because the overall map is a product of matrices.

In [ ]:
def compute_loss_and_grads(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    loss = 0.5 * np.mean(np.sum((acts[-1] - Y)**2, axis=0))
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return loss, grads

## Training Loop

The training loop runs for 300 steps with momentum-based updates.
For **SGD**: the momentum buffer accumulates raw gradients.
For **Muon**: the momentum buffer accumulates Newton-Schulz-orthogonalized gradients.

This means Muon's effective update direction is always on the tangent space of the
orthogonal group -- it "fixes the gauge" at every step. SGD, by contrast, can
accumulate arbitrary spectral distortions over training.

An early-stop check prevents NaN/divergence from corrupting results.

In [ ]:
def train(weights_init, X, Y, optimizer, lr):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss, grads = compute_loss_and_grads(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            break
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return weights

## Equivariance Violation Measurement

This is the key measurement function. For a given layer $\ell$:

1. **Train the original network** with Muon, obtaining trained weights $\{W_\ell^A\}$
2. **Conjugate** layer $\ell$'s initial weights: $W_\ell \to R\,W_\ell\,S^\top$ where $R, S \in O(n)$ are random orthogonal matrices
3. **Train the conjugated network** with Muon, obtaining $\{W_\ell^B\}$
4. **Measure violation** as:

$$\text{violation} = \frac{\|W_\ell^B - R\,W_\ell^A\,S^\top\|}{\|W_\ell^A\|}$$

If Muon were perfectly equivariant, this would be zero: conjugating the input
would produce a correspondingly conjugated output. Non-zero values indicate
that the data-dependent loss breaks this symmetry.

The violation is measured independently for each layer and each seed, then
correlated with the kappa-advantage to test whether gauge-breaking predicts
gauge-fixing benefit.

In [ ]:
def measure_equivariance_violation(weights_init, X, Y, target_layer, rng):
    R = random_orthogonal(DIM, rng)
    S = random_orthogonal(DIM, rng)

    weights_A = train([W.copy() for W in weights_init], X, Y, 'muon', LR_MUON)

    weights_conj = [W.copy() for W in weights_init]
    weights_conj[target_layer] = R @ weights_conj[target_layer] @ S.T
    weights_B = train(weights_conj, X, Y, 'muon', LR_MUON)

    expected = R @ weights_A[target_layer] @ S.T
    actual = weights_B[target_layer]
    return np.linalg.norm(actual - expected) / max(np.linalg.norm(weights_A[target_layer]), 1e-30)

## Main Experiment: Cross-Layer Correlation Analysis

The main function orchestrates the full experiment:

1. **Outer loop** over 5 random seeds -- each seed generates fresh data $(X, Y)$ and initial weights
2. **Inner loop** over 8 layers -- for each layer, we measure both equivariance violation and kappa advantage
3. **Aggregation**: average metrics across seeds, then compute Pearson correlation across layers

The kappa advantage is defined as:
$$\text{kappa\_advantage}_\ell = \frac{\kappa(W_\ell^{\text{SGD}})}{\kappa(W_\ell^{\text{Muon}})}$$

A value > 1 means Muon produces a better-conditioned weight matrix at that layer.
We then correlate this with the equivariance violation across all 8 layers.

In [ ]:
def main():
    print("=" * 100)
    print("2.1b-ii: EQUIVARIANCE VIOLATION vs PER-LAYER MUON ADVANTAGE")
    print("=" * 100)
    print(f"Network: {DEPTH}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Muon LR = {LR_MUON}, SGD LR = {LR_SGD}, Momentum = {MOMENTUM}")
    print(f"Newton-Schulz iterations = {NS_ITERS}")
    print()

    violations = {l: [] for l in range(DEPTH)}
    kappa_advantages = {l: [] for l in range(DEPTH)}

    for seed_idx in range(NUM_SEEDS):
        seed = 42 + seed_idx * 137
        rng = np.random.RandomState(seed)
        X = rng.randn(DIM, BATCH_SIZE) * 0.3
        Y = rng.randn(DIM, BATCH_SIZE) * 0.3
        weights_init = init_weights(seed + 5000)

        print(f"\n--- Seed {seed_idx + 1}/{NUM_SEEDS} (seed={seed}) ---")
        print(f"  Data: X shape={X.shape}, ||X||_F={np.linalg.norm(X):.3f}, ||Y||_F={np.linalg.norm(Y):.3f}")

        # Check initial condition numbers
        init_kappas = [np.linalg.cond(W) for W in weights_init]
        print(f"  Initial kappa range: [{min(init_kappas):.2f}, {max(init_kappas):.2f}]")

        # Train with both optimizers
        final_sgd = train([W.copy() for W in weights_init], X, Y, 'sgd', LR_SGD)
        final_muon = train([W.copy() for W in weights_init], X, Y, 'muon', LR_MUON)

        # Report final losses
        loss_sgd, _ = compute_loss_and_grads(final_sgd, X, Y)
        loss_muon, _ = compute_loss_and_grads(final_muon, X, Y)
        print(f"  Final loss -- SGD: {loss_sgd:.6e}, Muon: {loss_muon:.6e}")

        for l in range(DEPTH):
            # Kappa advantage
            svs_sgd = np.linalg.svd(final_sgd[l], compute_uv=False)
            svs_muon = np.linalg.svd(final_muon[l], compute_uv=False)
            kappa_sgd = svs_sgd[0] / max(svs_sgd[-1], 1e-12)
            kappa_muon = svs_muon[0] / max(svs_muon[-1], 1e-12)
            kappa_advantages[l].append(kappa_sgd / max(kappa_muon, 1e-12))

            # Equivariance violation
            rng_v = np.random.RandomState(seed + l * 1000)
            viol = measure_equivariance_violation(weights_init, X, Y, l, rng_v)
            violations[l].append(viol)

        print(f"  Per-layer kappa (SGD):  {['%.2f' % (np.linalg.svd(final_sgd[l], compute_uv=False)[0] / max(np.linalg.svd(final_sgd[l], compute_uv=False)[-1], 1e-12)) for l in range(DEPTH)]}")
        print(f"  Per-layer kappa (Muon): {['%.2f' % (np.linalg.svd(final_muon[l], compute_uv=False)[0] / max(np.linalg.svd(final_muon[l], compute_uv=False)[-1], 1e-12)) for l in range(DEPTH)]}")

    # Results
    print(f"\n{'=' * 100}")
    print("PER-LAYER RESULTS (averaged over seeds)")
    print(f"{'=' * 100}")

    print(f"\n  {'Layer':>6}  {'Violation':>12}  {'kappa ratio':>14}  {'Violation std':>14}")
    print("  " + "-" * 50)

    mean_viols = []
    mean_kappas = []
    for l in range(DEPTH):
        mv = np.mean(violations[l])
        mk = np.mean(kappa_advantages[l])
        mean_viols.append(mv)
        mean_kappas.append(mk)
        print(f"  {l:>6}  {mv:>12.4e}  {mk:>14.2f}x  {np.std(violations[l]):>14.4e}")

    print(f"\n  Violation range: [{min(mean_viols):.4e}, {max(mean_viols):.4e}]")
    print(f"  Kappa ratio range: [{min(mean_kappas):.2f}x, {max(mean_kappas):.2f}x]")
    print(f"  Mean violation across layers: {np.mean(mean_viols):.4e}")
    print(f"  Mean kappa ratio across layers: {np.mean(mean_kappas):.2f}x")

    corr = np.corrcoef(mean_viols, mean_kappas)[0, 1]
    print(f"\n  Pearson correlation(violation, kappa_advantage) = {corr:.3f}")

    # Rank-order correlation for robustness check
    from scipy.stats import spearmanr
    try:
        rho, p_val = spearmanr(mean_viols, mean_kappas)
        print(f"  Spearman rank correlation = {rho:.3f} (p = {p_val:.4f})")
    except ImportError:
        # Fall back if scipy not available
        rank_viols = np.argsort(np.argsort(mean_viols))
        rank_kappas = np.argsort(np.argsort(mean_kappas))
        rho = np.corrcoef(rank_viols, rank_kappas)[0, 1]
        print(f"  Spearman rank correlation (manual) = {rho:.3f}")

    # Hypothesis tests
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    t1 = corr > 0
    t3 = abs(corr) > 0.5

    print(f"\n  T1: Positive correlation (more violation -> more benefit)?")
    print(f"      r = {corr:.3f}")
    print(f"      --> {'PASS' if t1 else 'FAIL (negative correlation)'}")
    if t1:
        print(f"      Interpretation: Hypothesis A supported -- layers with more gauge-breaking")
        print(f"      benefit MORE from Muon's gauge-fixing, suggesting the violation measures")
        print(f"      genuine gauge drift that Muon corrects.")
    else:
        print(f"      Interpretation: Hypothesis B supported -- layers with more gauge-breaking")
        print(f"      benefit LESS from Muon, suggesting that violation reflects fundamental")
        print(f"      breakdown of the gauge-fixing mechanism rather than correctable drift.")

    print(f"\n  T3: Strong correlation (|r| > 0.5)?")
    print(f"      |r| = {abs(corr):.3f}")
    print(f"      --> {'PASS' if t3 else 'FAIL'}")
    if t3:
        print(f"      Interpretation: The relationship is strong enough to be mechanistically")
        print(f"      meaningful, not just noise.")
    else:
        print(f"      Interpretation: The correlation is weak, suggesting violation and advantage")
        print(f"      may be partially decoupled or confounded by other layer-dependent factors.")

    # Summary interpretation
    print(f"\n\n{'=' * 100}")
    print("SUMMARY INTERPRETATION")
    print(f"{'=' * 100}")
    if t1 and t3:
        print(f"  STRONG SUPPORT for Hypothesis A: equivariance violation positively and")
        print(f"  strongly predicts per-layer Muon advantage. This is consistent with the")
        print(f"  RG-gauge-fixing picture where violation measures gauge drift magnitude.")
    elif t1 and not t3:
        print(f"  WEAK SUPPORT for Hypothesis A: the correlation is positive but not strong.")
        print(f"  Other factors (layer position, gradient magnitude) may also influence advantage.")
    elif not t1 and t3:
        print(f"  SUPPORT for Hypothesis B: strong NEGATIVE correlation. Layers where")
        print(f"  equivariance is most broken see LESS Muon benefit.")
    else:
        print(f"  INCONCLUSIVE: weak or no correlation. The relationship between violation")
        print(f"  and advantage may be non-linear or dominated by other factors.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Tests

This experiment directly probes the **mechanistic link** between equivariance breaking
and optimizer benefit. If Muon works by gauge-fixing, then layers where the gauge
symmetry is most violated should be exactly the layers where gauge-fixing matters most.

### Interpretation Guide

- **T1 PASS + T3 PASS**: Strong evidence for the gauge-fixing interpretation. The violation
  is a meaningful diagnostic of where Muon's mechanism provides value.
- **T1 PASS + T3 FAIL**: The direction is right but the signal is weak. Other factors
  (layer depth, gradient scale, activation statistics) may dominate.
- **T1 FAIL**: Challenges Hypothesis A. Either the violation measures something different
  from what Muon corrects, or Muon's benefit comes from a mechanism other than
  correcting gauge drift (e.g., pure spectral regularization).

### Connection to the Broader Framework

This experiment connects to:
- **Exp 2.1b**: Established that equivariance violation exists and is layer-dependent
- **Exp 2.1b-i**: Showed violation scales with network depth
- **Exp 1.3a-i**: Measured effective rank layer-by-layer (related spectral diagnostic)
- **H3**: Normalized SGD vs Muon (alternative explanations for Muon's advantage)

A strong positive correlation here would strengthen the interpretation that Muon's
benefit is fundamentally about **gauge-fixing** rather than mere spectral regularization,
because it ties the benefit specifically to the gauge degree of freedom rather than
to generic spectral properties.